# Reinforcement Learning

# 2. Dynamic programming

This notebook presents policy iteration and value iteration for finding the optimal policy.

Note that these techniques require the enumeration of all states and thus apply to a few models only (e.g., walk, maze, Tic-Tac-Toe, Nim).

Please read the instructions:
* Write concise code and text.
* Check that your notebook runs without errors.
* Clear all cell outputs before upload on Moodle.

In [ ]:
import numpy as np

In [ ]:
from model import Walk, Maze, TicTacToe, Nim
from agent import Agent

## Walk

In [ ]:
walk = Walk()

In [ ]:
states = walk.get_all_states()

In [ ]:
len(states)

## Maze

In [ ]:
maze_map = np.load('maze.npy')

In [ ]:
maze = Maze()
init_state = (1, 0)
exit_state = (1, 20)
maze.set_parameters(maze_map, init_state, [exit_state])
maze = Maze()

In [ ]:
states = maze.get_all_states()

In [ ]:
len(states)

In [ ]:
maze.display()

## Policy Iteration

In policy iteration, you start from an arbitrary policy and improve it sequentially from its value function. The limiting policy is optimal.

In [ ]:
from dynamic import PolicyEvaluation, PolicyIteration

In [ ]:
# let's start with the random policy
agent = Agent(maze)
policy = agent.policy

In [ ]:
# policy evaluation
algo = PolicyEvaluation(maze, policy)
algo.evaluate_policy()
values = algo.values

In [ ]:
maze.display_values(values)

In [ ]:
# policy improvement
new_policy = algo.improve_policy()

In [ ]:
maze.display_policy(new_policy)

In [ ]:
# let's test this new policy
agent = Agent(maze, new_policy)
stop, states, rewards = agent.get_episode()

In [ ]:
animation = maze.display(states)

In [ ]:
animation

In general, several iterations of policy evaluation / policy improvement are necessary. 

In [ ]:
algo = PolicyIteration(maze, n_iter=5)

In [ ]:
policy = algo.get_optimal_policy()
values = algo.values
maze.display_values(values)

In [ ]:
maze.display_policy(policy)

## To do

* Do you see any difference in the policy after 1 step vs 5 steps of policy iteration?
* What is the length of the shortest path to the exit in the maze?

ANSWER:
After 1 step of policy iteration, the policy is not the best. Some actions are still inconsistent and the agent may take unnecessary detours.
After 5 steps of policy iteration, the policy has much improved. All arrows consistently point toward the goal. No useless detours remain.
Overall, the policy after 5 steps is clearly better and more stable than after 1 step.

ANSWER:
The length of the shortest path to the exit in the maze is 36. Indeed, it corresponds to the number of steps taken (states visted minus 1).

## To do

We now consider the walk with discount factor $\gamma = 0.9$.
* Display the value function after 5 steps of policy iteration. 
* Display the corresponding policy.
* What is the value of the initial state?
* Interpret the results obtained when the wind changes, as specified below. 

In [ ]:
walk = Walk()

In [ ]:
# no wind 
print(walk.Wind)

In [ ]:
algo = PolicyIteration(walk, n_iter=5, gamma=0.9)
policy = algo.get_optimal_policy()
values = algo.values
walk.display_values(values)

In [ ]:
walk.display_policy(policy)

In [ ]:
agent = Agent(walk, policy)
stop, states, rewards = agent.get_episode()
print(values[states[0]])

In [ ]:
# strong wind
wind = {(0, 1): 0.8}

In [ ]:
Walk.set_parameters(size=Walk.Size, rewards=Walk.Rewards, wind=wind)

In [ ]:
walk = Walk()

In [ ]:
algo = PolicyIteration(walk, n_iter=5, gamma=0.9)
policy = algo.get_optimal_policy()
values = algo.values
walk.display_values(values)

In [ ]:
walk.display_policy(policy)

In [ ]:
agent = Agent(walk, policy)
stop, states, rewards = agent.get_episode()
print(values[states[0]])

ANSWER:
Without wind, the agent moves exactly where it wants, so the best path is simple and direct.
With strong wind, the agent is sometimes pushed sideways. Because of this, the value function changes, and the policy adjusts to “fight” the wind and stay on track. The initial state’s value increases in this case because the wind pushes the agent closer to the goal.

## Value Iteration

Value iteration relies on Bellman's optimality equation. 

## To do

Check the code of ``ValueIteration`` below.
* Complete the method ``get_optimal_policy``.
* Test it on the walk and the maze.
* You play TicTacToe at random against an adversary using the one-step policy. What is your expected gain? 
* Observe the improvement when you play perfectly against the same adversary.

In [ ]:
class ValueIteration(PolicyEvaluation):
    """Value iteration. 
    
    Parameters
    ----------
    environment : object of class Environment
        Environment of the agent.
    player : int
        Player for games (1 or -1, default = default player of the game).        
    gamma : float
        Discount factor (between 0 and 1).
    n_iter : int
        Maximum number of value iterations.
    """
    
    def __init__(self, environment, player=None, gamma=1, n_iter=100):
        agent = Agent(environment, player=player)
        policy = agent.policy
        player = agent.player
        super(ValueIteration, self).__init__(environment, policy, player, gamma)  
        self.n_iter = n_iter
        
    def get_optimal_policy(self):
        """Get the optimal policy by iteration of Bellman's optimality equation."""
        transitions = self.transitions
        # Bellman's optimality equation
        values = np.zeros(self.n_states)
        for t in range(self.n_iter):
            next_values = self.rewards + self.gamma * values
            action_value = {action: transition.dot(next_values) for action, transition in self.transitions.items()}
            # new values, after iteration
            values = np.zeros(self.n_states)
            for i, state in enumerate(self.states):
                if not self.environment.is_terminal(state):
                    actions = self.get_actions(state)
                    values[i] = max(action_value[action][i] for action in actions)
                else:
                    values[i] = 0
        self.values = values
        policy = self.improve_policy()
        return policy

In [ ]:
walk = Walk()
algo = ValueIteration(walk)

policy = algo.get_optimal_policy()
values = algo.values

walk.display_values(values)
walk.display_policy(policy)

In [ ]:
maze = Maze()
algo = ValueIteration(maze)

policy = algo.get_optimal_policy()
values = algo.values

maze.display_values(values)
maze.display_policy(policy)

In [ ]:
game = TicTacToe()
algo = ValueIteration(game)

policy = algo.get_optimal_policy()
values = algo.values

In [ ]:
random_agent = Agent(game) 
gains_random = []

for _ in range(300):
    stop, states, rewards = random_agent.get_episode()
    gains_random.append(sum(rewards))

print("Expected gain when playing random:", np.mean(gains_random))

In [ ]:
optimal_agent = Agent(game, policy)
gains_optimal = []

for _ in range(300):
    stop, states, rewards = optimal_agent.get_episode()
    gains_optimal.append(sum(rewards))

print("Expected gain with optimal play:", np.mean(gains_optimal))

ANSWER:
With a random policy, the agent wins only occasionally (average gain: 0.33). With the optimal policy, the agent wins almost every game (average gain: 1.0), showing a clear improvement.

## Perfect players

We now use Value Iteration to get perfect players, assuming the best response of the adversary.

## To do

Check the code of the new class ``ValueIteration`` below.
* Complete the method ``get_perfect_players``.
* Test it on TicTacToe. Who wins?
* Test it on Nim. Who wins?
* Is this approach applicable to ConnectFour? Why?

In [ ]:
from scipy import sparse

In [ ]:
class ValueIteration(PolicyEvaluation):
    """Value iteration. 
    
    Parameters
    ----------
    environment : object of class Environment
        Environment of the agent.
    player : int
        Player for games (1 or -1, default = default player of the game).        
    gamma : float
        Discount factor (between 0 and 1).
    n_iter : int
        Maximum number of value iterations.
    """
    
    def __init__(self, environment, player=None, gamma=1, n_iter=100):
        agent = Agent(environment, player=player)
        policy = agent.policy
        player = agent.player
        super(ValueIteration, self).__init__(environment, policy, player, gamma)  
        self.n_iter = n_iter
    
    def get_perfect_players(self):
        """Get perfect players for games, with the best response of the adversary."""
        if not self.environment.is_game():
            raise ValueError("This method applies to games only.")
        # get transitions for each player
        actions = self.environment.get_all_actions()
        transitions = {action: sparse.lil_matrix((self.n_states, self.n_states)) for action in actions}
        for i, state in enumerate(self.states):    
            actions = self.environment.get_available_actions(state)
            for action in actions:
                next_state = self.environment.get_next_state(state, action)
                j = self.get_state_id(next_state)
                transitions[action][i, j] = 1
        transitions = {action: sparse.csr_matrix(transition) for action, transition in transitions.items()}
        self.transitions = transitions
        # Bellman's optimality equation
        values = np.zeros(self.n_states)
        for t in range(self.n_iter):
            next_values = self.rewards + self.gamma * values
            action_value = {action: transition.dot(next_values) for action, transition in transitions.items()}
            values = np.zeros(self.n_states)
            for i, state in enumerate(self.states):
                if not self.environment.is_terminal(state):
                    player, _ = state
                    actions = self.environment.get_available_actions(state)
                    # to be modified
                    if player == self.player:
                        values[i] = max(action_value[action][i] for action in actions)
                    else:
                        values[i] = min(action_value[action][i] for action in actions)  
                else:
                    values[i] = 0
        self.values = values

        # policies
        policy = self.improve_policy(self.player)
        adversary_policy = self.improve_policy(-self.player)
        return policy, adversary_policy
        

In [ ]:
Game = TicTacToe

In [ ]:
game = Game()

In [ ]:
algo = ValueIteration(game)

In [ ]:
policy, adversary_policy = algo.get_perfect_players()

In [ ]:
perfect_agent = Agent(game, policy, player=1)
adversary_agent = Agent(game, adversary_policy, player=-1)

results = []

for _ in range(300):
    stop, states, rewards = perfect_agent.get_episode()
    results.append(sum(rewards))

print(np.mean(results))

ANSWER:
For TicTacToe, the perfect player almost always wins. But, perfect play leads to a draw when both players are optimal.

In [ ]:
Game = Nim

In [ ]:
game = Game()

In [ ]:
algo = ValueIteration(game)

In [ ]:
policy, adversary_policy = algo.get_perfect_players()

In [ ]:
perfect_agent = Agent(game, policy, player=1)
adversary_agent = Agent(game, adversary_policy, player=-1)

results = []

for _ in range(300):
    stop, states, rewards = perfect_agent.get_episode()
    results.append(sum(rewards))

print(np.mean(results))

ANSWER:
For Nim, the perfect player wins.

ANSWER:

No, this approach is not applicable to ConnectFour.
Indeed, value iteration requires enumerating all possible states of the game, and ConnectFour has too many states:
* TicTacToe has 26,000 states, which is manageable
* Nim with small piles is manageable
* ConnectFour has 4.5 trillion states, which is astronomically large

Value Iteration requires enumerating every state and updating its value repeatedly, which is computationally impossible for a game of this size.